# Data Quality and Table Relationships

## Notebook purpose

This notebook validates the quality of the e-commerce datasets and confirms how the tables connect.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
DATA_DIR = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

Project root: C:\Users\ankur\OneDrive\Desktop\E-Commerce Project
Data directory: C:\Users\ankur\OneDrive\Desktop\E-Commerce Project\data


# Load Source Datasets

In [3]:
customers_df = pd.read_csv(DATA_DIR / "customer_master.csv")
orders_df = pd.read_csv(DATA_DIR / "ecommerce_sales_customer_analytics_150k.csv")
order_items_df = pd.read_csv(DATA_DIR / "order_items.csv")
products_df = pd.read_csv(DATA_DIR / "product_catalog.csv")
statistics_df = pd.read_csv(DATA_DIR / "dataset_statistics.csv")

datasets = {
    "customers": customers_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "products": products_df,
    "statistics": statistics_df,
}

print("All source datasets loaded successfully")

All source datasets loaded successfully


# Datasets Overview

In [6]:
overview_rows = []

for name, dataframe in datasets.items():
    overview_rows.append({
        "dataset": name,
        "rows": dataframe.shape[0],
        "columns": dataframe.shape[1],
    })

overview_df = pd.DataFrame(overview_rows)
display(overview_df)

,dataset,rows,columns
0,customers,25000,11
1,orders,138116,46
2,order_items,397569,12
3,products,1175,9
4,statistics,1,11


# Column names and data types

In [10]:
for name, dataframe in datasets.items():
    print(f"\n{name.upper()}")
    schema_df = pd.DataFrame({
        "column": dataframe.columns,
        "data_type": dataframe.dtypes.astype(str).values,
        "non_null_count": dataframe.notna().sum().values,
        "missing_count": dataframe.isna().sum().values
    })
    display(schema_df)


CUSTOMERS


,column,data_type,non_null_count,missing_count
0,customer_id,object,25000,0
1,customer_name,object,25000,0
2,customer_age,int64,25000,0
3,gender,object,25000,0
4,customer_segment,object,25000,0
5,customer_city,object,25000,0
6,customer_state,object,25000,0
7,customer_country,object,25000,0
8,region,object,25000,0
9,customer_postal_code,int64,25000,0



ORDERS


,column,data_type,non_null_count,missing_count
0,order_id,object,138116,0
1,order_date,object,138116,0
2,order_time,object,138116,0
3,order_status,object,138116,0
4,sales_channel,object,138116,0
5,customer_id,object,138116,0
6,customer_name,object,138116,0
7,customer_age,int64,138116,0
8,gender,object,138116,0
9,customer_segment,object,138116,0



ORDER_ITEMS


,column,data_type,non_null_count,missing_count
0,order_id,object,397569,0
1,product_id,object,397569,0
2,quantity,int64,397569,0
3,unit_price,float64,397569,0
4,discount_percentage,float64,397569,0
5,discount_amount,float64,397569,0
6,gross_sales,float64,397569,0
7,tax_amount,float64,397569,0
8,shipping_cost,float64,397569,0
9,net_sales,float64,397569,0



PRODUCTS


,column,data_type,non_null_count,missing_count
0,product_id,object,1175,0
1,product_name,object,1175,0
2,product_category,object,1175,0
3,product_subcategory,object,1175,0
4,brand,object,1175,0
5,supplier,object,1175,0
6,unit_price,float64,1175,0
7,product_cost,float64,1175,0
8,product_rating,float64,1175,0



STATISTICS


,column,data_type,non_null_count,missing_count
0,Total Transactions,int64,1,0
1,Total Columns,int64,1,0
2,Total Customers,int64,1,0
3,Total Products Used,int64,1,0
4,Date Range,object,1,0
5,Total Revenue,object,1,0
6,Total Profit,object,1,0
7,Average Order Value,object,1,0
8,Average Rating,float64,1,0
9,Return Rate,object,1,0


# Missing-value check


In [8]:
for name, dataframe in datasets.items():
    missing_df = (
        dataframe.isna()
        .sum()
        .reset_index()
        .rename(columns={"index": "column", 0: "missing_count"})
    )

    missing_df["missing_percentage"] = (
        missing_df["missing_count"] / len(dataframe) * 100
    ).round(2)

    missing_df = missing_df[missing_df["missing_count"] > 0]
    missing_df = missing_df.sort_values("missing_count", ascending=False)

    print(f"\n{name.upper()} - Missing values")

    if missing_df.empty:
        print("No missing values found.")
    else:
        display(missing_df)


CUSTOMERS - Missing values
No missing values found.

ORDERS - Missing values


,column,missing_count,missing_percentage
24,return_status,128654,93.15
25,return_reason,128654,93.15
31,coupon_code,110502,80.01
30,campaign_name,83233,60.26
21,delivery_days,24557,17.78
22,estimated_delivery_days,24557,17.78
26,customer_rating,24557,17.78
27,review_sentiment,24557,17.78
28,customer_review,24557,17.78



ORDER_ITEMS - Missing values
No missing values found.

PRODUCTS - Missing values
No missing values found.

STATISTICS - Missing values
No missing values found.


# Duplicate and primary-key checks

In [11]:
key_checks = pd.DataFrame([
    {
        "dataset": "customers",
        "key_column": "customer_id",
        "total_rows": len(customers_df),
        "unique_key_values": customers_df["customer_id"].nunique(),
        "duplicate_key_rows": customers_df["customer_id"].duplicated().sum()
    },
    {
        "dataset": "orders",
        "key_column": "order_id",
        "total_rows": len(orders_df),
        "unique_key_values": orders_df["order_id"].nunique(),
        "duplicate_key_rows": orders_df["order_id"].duplicated().sum()
    },
    {
        "dataset": "products",
        "key_column": "product_id",
        "total_rows": len(products_df),
        "unique_key_values": products_df["product_id"].nunique(),
        "duplicate_key_rows": products_df["product_id"].duplicated().sum()
    }
])

display(key_checks)

,dataset,key_column,total_rows,unique_key_values,duplicate_key_rows
0,customers,customer_id,25000,25000,0
1,orders,order_id,138116,138116,0
2,products,product_id,1175,1175,0


# Date Validation

In [13]:
orders_df["order_date"] = pd.to_datetime(
    orders_df["order_date"],
    errors="coerce"
)

date_summary = {
    "minimum_order_date": orders_df["order_date"].min(),
    "maximum_order_date": orders_df["order_date"].max(),
    "missing_or_invalid_dates": orders_df["order_date"].isna().sum(),
    "unique_order_dates": orders_df["order_date"].nunique()
}

display(pd.Series(date_summary))

minimum_order_date          2021-01-01 00:00:00
maximum_order_date          2025-12-31 00:00:00
missing_or_invalid_dates                      0
unique_order_dates                         1826
dtype: object

In [14]:
orders_df.head()

,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,customer_type,customer_city,customer_state,customer_country,region,customer_postal_code,payment_method,payment_status,currency,shipping_method,warehouse,delivery_days,estimated_delivery_days,delivery_status,return_status,return_reason,customer_rating,review_sentiment,customer_review,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,quantity,gross_sales,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,Loyal,Lake Williamberg,Texas,USA,South,86040,Digital Wallet,Paid,USD,Standard,WH-003,3.00,3.00,On Time,NaN,NaN,3.50,Positive,Satisfied with the purchase.,Direct,Default_Campaign,NaN,94,44,5,"1,350.19",477.89,61.07,12.03,945.40,588.33,345.04,36.50,"12,459.68",True,11
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,Loyal,Kristyport,Baden-Württemberg,Germany,South,62538,Debit Card,Pending,EUR,Economy,WH-005,10.00,10.00,On Time,NaN,NaN,3.60,Positive,Good value for money.,Direct,Default_Campaign,NaN,201,171,8,"3,144.64","1,456.06",320.83,9.00,"2,018.41","2,031.88",-22.47,-1.11,"13,032.48",True,11
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,Loyal,Singletonhaven,New York,USA,East,19993,Cash on Delivery,Paid,USD,Express,WH-015,3.00,3.00,On Time,NaN,NaN,4.30,Positive,Good product. Works as expected.,YouTube,NaN,NaN,46,16,5,522.81,97.30,29.79,13.05,468.35,257.73,197.57,42.18,"6,159.44",True,6
3,ORD-567636,2023-01-21,07:20:26,Completed,Social Media,CUST-023928,Jennifer Smith,65,Male,Consumer,Loyal,Wigginsstad,North Carolina,USA,South,66329,Debit Card,Paid,USD,Express,WH-010,2.00,2.00,On Time,NaN,NaN,3.50,Positive,Satisfied with the purchase.,Email Marketing,NaN,NaN,54,10,2,544.62,53.34,34.39,20.85,546.52,277.84,247.83,45.35,"5,638.30",True,7
4,ORD-820028,2022-05-13,09:46:21,Completed,Social Media,CUST-012730,Jessica Wang,34,Female,Consumer,Loyal,New Michaelton,Gujarat,India,West,77306,Digital Wallet,Paid,INR,Standard,WH-013,6.00,5.00,Delayed,NaN,NaN,3.00,Neutral,Mixed feelings about this purchase.,Direct,NaN,NaN,278,147,6,"2,474.52",133.63,421.35,23.03,"2,785.27","1,231.88","1,530.36",54.94,"13,240.10",True,8


# Return label validation

In [17]:
return_distribution = (
    orders_df["return_status"]
    .fillna("Not Returned")
    .value_counts(dropna=False)
    .rename_axis("return_status")
    .reset_index(name="order_count")
)

return_distribution["percentage"] = (
    return_distribution["order_count"] / len(orders_df) * 100
).round(2)

display(return_distribution)

,return_status,order_count,percentage
0,Not Returned,128654,93.15
1,Returned,9462,6.85


# Customer-to-order relationship check


In [18]:
order_customer_ids = set(orders_df["customer_id"])
master_customer_ids = set(customers_df["customer_id"])

orders_without_customer = order_customer_ids - master_customer_ids
customers_without_order = master_customer_ids - order_customer_ids

customer_join_check = pd.Series({
    "unique_customers_in_orders": len(order_customer_ids),
    "unique_customers_in_master": len(master_customer_ids),
    "order_customers_not_in_master": len(orders_without_customer),
    "master_customers_without_orders": len(customers_without_order)
})

display(customer_join_check)

unique_customers_in_orders         24911
unique_customers_in_master         25000
order_customers_not_in_master          0
master_customers_without_orders       89
dtype: int64

# Order-item relationship check

In [19]:
order_ids = set(orders_df["order_id"])
item_order_ids = set(order_items_df["order_id"])

order_item_check = pd.Series({
    "unique_orders_in_orders_table": len(order_ids),
    "unique_orders_in_items_table": len(item_order_ids),
    "item_orders_not_in_orders": len(item_order_ids - order_ids),
    "orders_without_items": len(order_ids - item_order_ids),
    "average_items_per_order": round(
        len(order_items_df) / len(item_order_ids), 2
    )
})

display(order_item_check)

unique_orders_in_orders_table   138,116.00
unique_orders_in_items_table    138,116.00
item_orders_not_in_orders             0.00
orders_without_items                  0.00
average_items_per_order               2.88
dtype: float64

# Product catalog relationship check

In [20]:
item_product_ids = set(order_items_df["product_id"])
catalog_product_ids = set(products_df["product_id"])

product_join_check = pd.Series({
    "unique_products_in_order_items": len(item_product_ids),
    "unique_products_in_catalog": len(catalog_product_ids),
    "item_products_not_in_catalog": len(item_product_ids - catalog_product_ids),
    "catalog_products_not_used_in_orders": len(catalog_product_ids - item_product_ids)
})

display(product_join_check)

unique_products_in_order_items         1175
unique_products_in_catalog             1175
item_products_not_in_catalog              0
catalog_products_not_used_in_orders       0
dtype: int64

# Conclusion

In [21]:
quality_summary = pd.DataFrame([
    {
        "check": "Customer IDs are unique",
        "result": customers_df["customer_id"].duplicated().sum() == 0
    },
    {
        "check": "Order IDs are unique",
        "result": orders_df["order_id"].duplicated().sum() == 0
    },
    {
        "check": "Product IDs are unique",
        "result": products_df["product_id"].duplicated().sum() == 0
    },
    {
        "check": "Orders have valid customers",
        "result": len(orders_without_customer) == 0
    },
    {
        "check": "Order items have valid orders",
        "result": len(item_order_ids - order_ids) == 0
    },
    {
        "check": "Order items have valid products",
        "result": len(item_product_ids - catalog_product_ids) == 0
    },
    {
        "check": "Order dates are valid",
        "result": orders_df["order_date"].isna().sum() == 0
    }
])

quality_summary["status"] = np.where(
    quality_summary["result"],
    "PASS",
    "REVIEW"
)

display(quality_summary[["check", "status"]])

,check,status
0,Customer IDs are unique,PASS
1,Order IDs are unique,PASS
2,Product IDs are unique,PASS
3,Orders have valid customers,PASS
4,Order items have valid orders,PASS
5,Order items have valid products,PASS
6,Order dates are valid,PASS
